# MSTL post hoc threshold optimization - excluding misgendering

The fitted model is frozen before this notebook begins. Accuracy- and fairness-oriented threshold pairs are selected from official training-set predictions only (prespecified ΔFPR constraint: 0.07), then applied without modification to held-out testing predictions.


In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from threshold_protocol import evaluate_frozen_thresholds, select_threshold_pairs

MODEL_SLUG = "mstl"
OUTCOME_SLUG = "excluding_misgendering"
MAX_FPR_GAP = 0.07
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)

prediction_dir = RESULTS_DIR / "predictions"
training_prediction_file = prediction_dir / f"{MODEL_SLUG}_{OUTCOME_SLUG}_training.csv"
testing_prediction_file = prediction_dir / f"{MODEL_SLUG}_{OUTCOME_SLUG}_testing.csv"

# Each file is produced by the same frozen fitted model and must contain:
# label (0/1), GEP (0=NGEP, 1=GEP), and probability (positive-class probability).
required_columns = {"label", "GEP", "probability"}

# Threshold selection uses training predictions only.
training_predictions = pd.read_csv(training_prediction_file)
missing = required_columns.difference(training_predictions.columns)
if missing:
    raise KeyError(f"Training predictions are missing columns: {sorted(missing)}")

selected_on_training = select_threshold_pairs(
    training_labels=training_predictions["label"].astype(int),
    training_probabilities=training_predictions["probability"].astype(float),
    training_groups=training_predictions["GEP"].astype(int),
    threshold_grid=THRESHOLD_GRID,
    max_fpr_gap=MAX_FPR_GAP,
)

threshold_dir = RESULTS_DIR / "thresholds"
threshold_dir.mkdir(parents=True, exist_ok=True)
threshold_file = threshold_dir / f"{MODEL_SLUG}_{OUTCOME_SLUG}_thresholds.json"
threshold_file.write_text(
    json.dumps({name: result.to_dict() for name, result in selected_on_training.items()}, indent=2),
    encoding="utf-8",
)

# Only now are the held-out testing predictions opened. The threshold pairs are frozen.
testing_predictions = pd.read_csv(testing_prediction_file)
missing = required_columns.difference(testing_predictions.columns)
if missing:
    raise KeyError(f"Testing predictions are missing columns: {sorted(missing)}")

heldout_results = evaluate_frozen_thresholds(
    testing_labels=testing_predictions["label"].astype(int),
    testing_probabilities=testing_predictions["probability"].astype(float),
    testing_groups=testing_predictions["GEP"].astype(int),
    selected_on_training=selected_on_training,
)

heldout_table = pd.DataFrame(
    [{"configuration": name, **result.to_dict()} for name, result in heldout_results.items()]
)
heldout_table.to_csv(
    RESULTS_DIR / f"{MODEL_SLUG}_{OUTCOME_SLUG}_heldout_threshold_results.csv",
    index=False,
)
print("Thresholds selected on training predictions:")
display(pd.DataFrame([{"configuration": name, **result.to_dict()} for name, result in selected_on_training.items()]))
print("Final held-out testing results (thresholds applied unchanged):")
display(heldout_table)
